# Classical Traffic Sign and Traffic Light Detection



In [4]:
from pathlib import Path
import json
from html import escape
from IPython.display import HTML, display

ROOT = Path.cwd()
results = json.loads((ROOT / "assets/test_results.json").read_text(encoding="utf-8"))

def show_table(headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


## 1. Objective and Dataset

The objective of this project is to develop a classical computer vision system that localizes traffic signs and traffic lights in street images. For each detected traffic light, the system also assigns a color: `red`, `yellow`, `green`, or `unknown`.

The project uses BDD100K images and their bounding box annotations. Each annotation provides the object category and its location in the original image. Additional attributes describe conditions such as weather and time of day. These attributes support the analysis of challenging examples.

The dataset contains three splits:

| Split | Available images | Use in this project |
|---|---:|---|
| Training | 70,000 | Source images for positive and negative training samples |
| Validation | 10,000 | Parameter selection; a subset of 1,000 images was used for final threshold selection |
| Test | 20,000 | Final evaluation with frozen models and parameters |

Initial experiments were conducted on subsets of the training and validation data to assess the feasibility of candidate methods before scaling up the selected approach.

The target output classes are:

- `traffic_sign`
- `traffic_light_red`
- `traffic_light_yellow`
- `traffic_light_green`
- `traffic_light_unknown`

The `unknown` label indicates that the light color is unspecified in the annotation. It does not necessarily indicate an unlit traffic light.

Detection is evaluated separately for `traffic_sign` and the combined `traffic_light` category using precision, recall, and F1 at an intersection over union threshold of **IoU >= 0.5**. Color classification is evaluated separately, using only correctly localized traffic lights.



## 2. Dataset Analysis

We analyzed the 70,000 training images.



### 2.1. Class Distribution

The table shows object counts and image counts for each class. One image can contain several objects and classes.

| Class | Objects | Images containing the class |
|---|---:|---:|
| `traffic_sign` | 239,893 | 57,242 |
| `traffic_light_red` | 46,157 | 18,662 |
| `traffic_light_yellow` | 3,418 | 2,049 |
| `traffic_light_green` | 79,449 | 29,218 |
| `traffic_light_unknown` | 57,200 | 20,401 |

Invalid boxes are excluded from these counts.

![Training object counts and image counts by class](assets/figure_01_class_counts.png)

*Figure 1. Objects and images by class.*

Signs are the largest class. Green lights are much more common than yellow lights. We therefore report results for each color, since overall accuracy can hide poor results for rare classes.



### 2.2. Object Sizes and Aspect Ratios

We measured object size using the shorter side of its box in the original image. The most common range is 8–15 pixels. Most objects have a shorter side below 32 pixels.

![Distribution of bounding box minimum side lengths](assets/figure_02_object_sizes.png)

*Figure 2. Shorter box side in pixels. Light colors are grouped together.*

Traffic sign sizes and aspect ratios:

| Percentile | Width (px) | Height (px) | Aspect ratio (width / height) |
|---|---:|---:|---:|
| 5th | 10.12 | 7.71 | 0.53 |
| 25th | 16.22 | 12.81 | 0.85 |
| 50th | 23.95 | 18.81 | 1.17 |
| 75th | 37.81 | 30.34 | 1.96 |
| 95th | 79.60 | 63.70 | 3.33 |

Percentiles are calculated separately for each column. A row does not describe one box.

Small objects contain little detail. Resizing them does not add missing detail, and a box shifted by a few pixels can have a low IoU. Box shapes also vary, so we tested different window shapes.

![Traffic sign aspect ratio distribution](assets/figure_03_aspect_ratios.png)

*Figure 3. Traffic sign box aspect ratios.*



### 2.3. Lighting and Weather Conditions

Daytime images are the largest group, followed by night images. Clear weather is the most common, while fog is rare. Missing condition labels are shown as `undefined`.

![Training images and objects by time of day](assets/figure_04_classes_by_timeofday.png)

*Figure 4. Images and objects by time of day.*

![Training images and objects by weather](assets/figure_05_classes_by_weather.png)

*Figure 5. Images and objects by weather.*

We also displayed reference boxes on images from different conditions.

![Reference annotations in clear daytime conditions](assets/figure_06_aaf122ed-78a30768_gt.jpg)

*Figure 6. Clear daytime: small objects against trees and buildings.*

![Reference annotations in rainy daytime conditions](assets/figure_07_7f9547ee-6d277ccc_gt.jpg)

*Figure 7. Rainy daytime: drops on the windshield and many small objects.*

![Reference annotations in rainy nighttime conditions](assets/figure_08_47459d96-00e0ffee_gt.jpg)

*Figure 8. Rainy night: bright lamps and dark traffic light bodies.*

Dark areas can hide object edges. Lamps and reflections can create false detections. These examples show possible problems, they do not measure accuracy for each condition.



### 2.4. Annotation Quality and Exceptional Cases

We checked box coordinates, sizes, and image boundaries. We found these cases:

- **Invalid boxes:** boxes with zero width or height were excluded from statistics and training samples.
- **Very small boxes:** some sign boxes have a width or height below one pixel. We kept valid boxes regardless of size.
- **Hidden or cut-off objects:** some objects are partly hidden or cut off by the image border. We kept these objects.
- **Unknown light colors:** the color is not specified in the label, even if a colored area is visible.

We inspected selected images, not every image. The checks do not guarantee that all images and labels are correct.

![Training image with a partly hidden sign](assets/figure_09_00067cfb-caba8a02.jpg)

*Figure 9. The green road sign near the center is marked as partly hidden (`occluded=true`). Only part of its appearance is visible.*

![Training image with a sign cut off by the image border](assets/figure_10_0000f77c-6257be58.jpg)

*Figure 10. The sign in the top-left corner is cut off by the image border (`truncated=true`). Its full shape is not visible.*



## 3. Method Development



### 3.1. Initial Idea: HOG, SVM, and Sliding Windows

We started with traffic signs. Since the dataset provides reference boxes, we could use them to extract image patches containing signs. The idea was to train a classifier that decides whether a patch contains a sign or background.

Each patch had to be converted into a fixed-length feature vector. We chose HOG (Histogram of Oriented Gradients), which describes local edge directions. These features capture parts of the object's shape and visual structure. We then used a linear SVM to learn the difference between sign and background features.

Training required both positive and negative examples. Positive patches contained annotated signs. Negative patches came from image regions without signs. Choosing useful background examples was an important part of the design, since the classifier would later see many types of background.

To locate signs in a full image, we chose a sliding window. The window moves across the image, and the classifier scores each patch. Windows with high scores become candidate detections. During detection, the reference boxes are not used.

Signs appear at different sizes, so one fixed window on the original image would cover only some of them. We therefore planned to scan resized versions of the whole image. Window proportions also had to account for both tall and wide signs.

This gave us three main choices to investigate: window shapes and sizes, image scales, and the selection of positive and negative training patches.



### 3.2. Window Shapes and Training Patches

We compared window shapes using the training boxes. A square window could reach IoU >= 0.5 for about 78.6% of signs under ideal placement and size. Combining tall and wide windows with width-to-height ratios of 0.5 and 2 increased this to 98.7%. This was a geometric estimate, not detection recall.

![Geometric coverage for combinations of window aspect ratios](assets/figure_11_combinations_coverage.png)

*Figure 11. Training signs that can reach IoU >= 0.5 for each combination of window ratios, assuming ideal placement and size.*

The pair 0.5 + 2 gave the best coverage among the tested pairs. We chose two window shapes as a balance between coverage and the amount of scanning.

For this sliding window approach, we trained two separate SVMs: one for tall windows and one for wide windows. Each used HOG features from its own window shape. We tested several patch sizes and used 32 × 64 and 64 × 32 pixels in the following detection experiments.

Each positive patch contained the full sign and some original background where needed to fit the window shape. Each sign was assigned to the better-fitting branch. Negative patches were sampled from regions with no overlap with annotated signs.



### 3.3. Sliding Window Results

The first full-image results produced many false detections. Background patches around cars, lamps, and walls received high SVM scores.

![Early sliding window detections with score threshold 0](assets/figure_12_detections.jpg)

*Figure 12. Early sliding window result at score threshold 0. Many background regions are marked as signs.*

Raising the score threshold removed most boxes, but false detections remained. The visible yield sign was also missed.

![Early sliding window detections with score threshold 2](assets/figure_13_detections.jpg)

*Figure 13. The same models and image at score threshold 2. Fewer boxes remain, but the yield sign is not detected.*

We tried to improve the classifier with hard negative mining. We collected background patches that the detector marked as signs and added them as negative training examples. In the example below, the retrained model returned no detections at threshold 2, including no detection of the yield sign.

![Sliding window result after hard negative mining](assets/figure_14_detections.jpg)

*Figure 14. After hard negative mining, at score threshold 2. No boxes are returned, but the visible sign is missed.*

We also added HSV color features to HOG. This gave the SVM information about both edges and colors, but many background patches still received high scores.

![Sliding window result with HOG and HSV features](assets/figure_15_detections.jpg)

*Figure 15. HOG + HSV at score threshold 0. Many false detections remain.*

Sliding windows tested many background patches at different positions and scales. Even a small share of false positives could produce many wrong boxes. Raising the threshold also removed real signs, so our version did not give a useful balance. We decided to change how candidate locations were selected.



### 3.4. Finding Candidate Regions with MSER

The next idea was to find promising image regions first and classify only those regions. This could reduce the number of background patches passed to the classifier.

We chose MSER (Maximally Stable Extremal Regions). It finds connected areas that stay similar in size as the brightness threshold changes. Such areas can match parts of a sign, such as its background or letters. We applied MSER to a grayscale image and a color map based on the relative amounts of red and blue.

Each region gave us a candidate bounding box. We could then extract the patch, calculate its features, and classify it as a sign or background. These boxes followed image regions, so we no longer needed two fixed window shapes for scanning.

MSER itself does not recognize signs. It can also find background regions or only a small part of a sign. We therefore first checked how many reference signs had a candidate box with IoU >= 0.5.



### 3.5. Larger Boxes and Canny Edges

On 20 validation images with 67 signs, MSER covered 22 signs. Some boxes were too small, so we added versions with width and height multiplied by 1.25, 1.5, and 2. Together with the original boxes, they covered 30 signs.

We then added Canny edge detection. We joined nearby edges and found contours to create more candidate boxes. Combining these boxes with MSER covered 42 signs, including 12 missed by MSER alone.

![Comparison of proposal coverage](assets/figure_16_proposal_coverage.png)

*Figure 16. Proposal coverage on the same 20 validation images. This measures available boxes before classification, not final detection recall.*

Better coverage came with more boxes: about 5,243 per image for expanded MSER and 8,519 for MSER + Canny. The classifier still had to reject most of them.



### 3.6. Choosing the Classifier

We trained classifiers on the proposed image patches using HOG, HSV, and box shape and size features. We then compared the linear SVM with Random Forest on the same MSER + Canny training samples. Random Forest combines many decision trees and can learn more complex rules from these features.

![Comparison of detection F1 during development](assets/figure_17_detection_comparison.png)

*Figure 17. Development results on the same 20 validation images. Each model uses its own best validation threshold. Earlier stages also used different training samples and features.*

With MSER + Canny, Random Forest improved F1 from 11.87% to 17.43%. We selected it for the final system. These small validation experiments guided our choices; the final test results are reported separately.

![Random Forest precision-recall curve](assets/figure_18_precision_recall.png)

*Figure 18. Precision and recall at different score thresholds. The red point marks the best validation F1.*

The selected approach was **MSER + Canny → HOG, HSV, and geometry → Random Forest → NMS**. NMS removes overlapping detections, keeping the higher-scoring boxes.

Developing the sign detector took much of the available time. We therefore reused the same approach for traffic lights and trained a separate Random Forest on traffic light examples. We then added a module to recognize the light color.



## 4. Final Method



### 4.1. Detection

1. Find boxes with MSER + Canny and add larger versions.
2. Resize patches to 32 × 32 pixels and calculate 407 HOG, HSV, and geometry features.
3. Classify patches with a separate Random Forest for signs or lights.
4. Apply a score threshold and NMS at IoU 0.3 to remove overlapping boxes.

Output boxes use the original image coordinates.



### 4.2. Training and Thresholds

We generated samples from all 70,000 training images. Positive boxes had IoU >= 0.5 with a target; negative boxes had IoU < 0.2 with every target.

Each model used 100,000 positive and 200,000 negative samples, with 200 trees and seed 42. We chose the best F1 threshold on 1,000 validation images:

| Detector | Score threshold | Validation F1 |
|---|---:|---:|
| Traffic signs | 0.8963 | 18.90% |
| Traffic lights | 0.8991 | 21.34% |

All settings were frozen before testing.



### 4.3. Traffic Light Color

We used HSV to find bright red, yellow, or green areas inside each detected light. The strongest color was selected if it clearly beat the others. Otherwise, or when no color was found, the result was `unknown`.



## 5. Test Results



### 5.1. Detection

We tested both models on 20,000 images. A correct detection needed IoU >= 0.5, with one prediction per reference box. Counts were summed across images.



In [5]:
headers = ["Detector", "Reference objects", "TP", "FP", "FN", "Precision", "Recall", "F1"]
rows = []
for row in results["detection"]:
    rows.append([row["class"], row["ground_truth_objects"], row["true_positive"], row["false_positive"], row["false_negative"], f"{row['precision_percent']:.2f}%", f"{row['recall_percent']:.2f}%", f"{row['f1_percent']:.2f}%"])
show_table(headers, rows)


Detector,Reference objects,TP,FP,FN,Precision,Recall,F1
traffic_sign,68992,12014,50964,56978,19.08%,17.41%,18.21%
traffic_light,52817,11773,45443,41044,20.58%,22.29%,21.40%


![Final detection results](assets/figure_19_test_detection.png)

*Figure 19. Test detection results.*

Both detectors still make many mistakes. Mean compute time per image was 4.51 s for signs and 3.50 s for lights, excluding file saving.



### 5.2. Color Classification

Color was correct for 9,887 of 11,773 correctly detected lights: **83.98% accuracy**.



In [6]:
headers = ["Reference color", "Matched lights", "Precision", "Recall", "F1"]
rows = []
for row in results["color"]["per_class"]:
    rows.append([row["color"], row["support"], f"{100 * row['precision']:.2f}%", f"{100 * row['recall']:.2f}%", f"{100 * row['f1']:.2f}%"])
show_table(headers, rows)


Reference color,Matched lights,Precision,Recall,F1
red,3272,95.17%,90.40%,92.73%
yellow,189,11.18%,50.79%,18.32%
green,7851,98.53%,86.32%,92.02%
unknown,461,6.03%,12.15%,8.06%


![Traffic light color confusion matrix](assets/figure_20_test_color_confusion.png)

*Figure 20. Color errors on correctly detected lights. Rows: reference; columns: prediction.*

Red and green work best and make up most examples. Yellow and unknown remain difficult, so overall accuracy does not tell the full story.



## 6. Error Analysis

Green boxes are references. Red marks the selected error. The captions give possible causes.



### 6.1. Traffic Signs

![Traffic sign false positive: box too large](assets/figure_21_false_positive_1_df333876-e81b53ab.jpg)

*Figure 21. FP: the box is too large. Better box fitting could help.*

![Traffic sign false positive: only part of a sign](assets/figure_22_false_positive_2_dd336573-28a7eeae.jpg)

*Figure 22. FP: only part of the sign is found. Joining nearby regions could help.*

![Traffic sign false negative near the image border](assets/figure_23_false_negative_1_d3a13316-faae99d4.jpg)

*Figure 23. FN: a small, low-contrast sign near the border is missed.*

![Traffic sign false negative: small distant object](assets/figure_24_false_negative_2_ccb3ee5b-bcee0539.jpg)

*Figure 24. FN: a distant 19 × 14 pixel sign has little detail.*



### 6.2. Traffic Lights

![Traffic light false positive: incomplete box](assets/figure_25_false_positive_1_d3a13316-faae99d4.jpg)

*Figure 25. FP: the box is close to the light, but IoU is below 0.5.*

![Traffic light false positive: unmatched bright region](assets/figure_26_false_positive_2_e1f25fca-ecf16e21.jpg)

*Figure 26. FP: an unmatched bright region. Harder negative samples could help.*

![Traffic light false negative: dark light body](assets/figure_27_false_negative_1_d3a13316-faae99d4.jpg)

*Figure 27. FN: the lamp is visible, but its dark body is hard to separate.*

![Traffic light false negative: distant lights](assets/figure_28_false_negative_2_e1f25fca-ecf16e21.jpg)

*Figure 28. FN: a distant 8 × 9 pixel light is missed.*



## 7. Conclusions

MSER + Canny + Random Forest worked best in our experiments. Small objects, poor box fitting, and background confusion remain the main problems. Color works better, mainly for red and green.

Merging nearby candidate regions and refining their boundaries could improve box fit and reduce localization errors. Adding hard negative examples such as lamps and reflections to training could help the classifier reject similar background regions.

We did not tune the system on test results. Predictions are saved for Laboratory 2B.
